# Adaptive Road Vision — Environment & GPU Check

This notebook verifies the Python environment, PyTorch/CUDA, NVIDIA GPU, GPU memory, AMP support, project folders, and BDD100K paths before training.

## 1. Project Configuration

Configured for your Windows project folder:
`C:\Users\ankit\Desktop\Adaptive_Road_Vision`

In [1]:
from pathlib import Path
import os, sys, platform, subprocess, shutil, time

PROJECT_ROOT = Path(r"C:\\Users\\ankit\\Desktop\\Adaptive_Road_Vision")
print("Project root:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())
print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])

Project root: C:\Users\ankit\Desktop\Adaptive_Road_Vision
Exists: True
Python executable: c:\Users\ankit\AppData\Local\Programs\Python\Python312\python.exe
Python version: 3.12.10


## 2. Operating System & Python Information

In [2]:
print("=" * 70)
print("SYSTEM INFORMATION")
print("=" * 70)
print("OS          :", platform.system(), platform.release())
print("Machine     :", platform.machine())
print("Processor   :", platform.processor())
print("Python      :", platform.python_version())
print("Python path :", sys.executable)
print("Working dir :", Path.cwd())
print("nvidia-smi  :", "AVAILABLE" if shutil.which("nvidia-smi") else "NOT FOUND")

SYSTEM INFORMATION
OS          : Windows 11
Machine     : AMD64
Processor   : Intel64 Family 6 Model 154 Stepping 3, GenuineIntel
Python      : 3.12.10
Python path : c:\Users\ankit\AppData\Local\Programs\Python\Python312\python.exe
Working dir : c:\Users\ankit\Desktop\Adaptive_Road_Vision\notebooks
nvidia-smi  : AVAILABLE


## 3. Core Package Check

This cell only reports installed packages; it does not install anything.

In [3]:
import importlib.util
packages = {"torch":"torch", "torchvision":"torchvision", "numpy":"numpy", "pandas":"pandas", "opencv":"cv2", "Pillow":"PIL", "PyYAML":"yaml", "matplotlib":"matplotlib", "scikit-learn":"sklearn"}
for label, module in packages.items():
    print(f"{label:15} : {'INSTALLED' if importlib.util.find_spec(module) else 'MISSING'}")

torch           : INSTALLED
torchvision     : INSTALLED
numpy           : INSTALLED
pandas          : INSTALLED
opencv          : INSTALLED
Pillow          : INSTALLED
PyYAML          : INSTALLED
matplotlib      : INSTALLED
scikit-learn    : INSTALLED


## 4. PyTorch & CUDA Check

In [4]:
try:
    import torch
    print("=" * 70)
    print("PYTORCH / CUDA")
    print("=" * 70)
    print("PyTorch version      :", torch.__version__)
    print("Torch CUDA available :", torch.cuda.is_available())
    print("Torch CUDA version   :", torch.version.cuda)
    print("cuDNN version        :", torch.backends.cudnn.version())
    print("GPU count            :", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("Default GPU          :", torch.cuda.get_device_name(0))
        print("Compute capability   :", torch.cuda.get_device_capability(0))
    else:
        print("WARNING: CUDA is not available in this Jupyter environment.")
except ImportError:
    print("PyTorch is not installed in this Jupyter kernel.")

PYTORCH / CUDA
PyTorch version      : 2.14.0+cpu
Torch CUDA available : False
Torch CUDA version   : None
cuDNN version        : None
GPU count            : 0


## 5. Detailed NVIDIA GPU Information

In [5]:
def run_command(command):
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=15)
        print(result.stdout if result.stdout else result.stderr)
        return result.returncode
    except Exception as e:
        print("Command failed:", e)
        return -1

if shutil.which("nvidia-smi"):
    run_command(["nvidia-smi", "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free,temperature.gpu,utilization.gpu", "--format=csv"])
else:
    print("nvidia-smi was not found. Verify the NVIDIA driver if an NVIDIA GPU is installed.")

index, name, driver_version, memory.total [MiB], memory.used [MiB], memory.free [MiB], temperature.gpu, utilization.gpu [%]
0, NVIDIA GeForce RTX 2050, 592.82, 4096 MiB, 0 MiB, 3962 MiB, 52, 0 %



## 6. GPU Memory Check

In [6]:
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}")
        print(f"  Total VRAM : {props.total_memory / (1024**3):.2f} GB")
        print(f"  Allocated  : {torch.cuda.memory_allocated(i) / (1024**3):.2f} GB")
        print(f"  Reserved   : {torch.cuda.memory_reserved(i) / (1024**3):.2f} GB")
        print(f"  Compute    : {props.major}.{props.minor}")
else:
    print("Skipped: CUDA unavailable.")

Skipped: CUDA unavailable.


## 7. CUDA Tensor Test

This confirms that PyTorch can allocate and compute on the GPU.

In [7]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    x = torch.randn((2048, 2048), device=device)
    y = torch.randn((2048, 2048), device=device)
    torch.cuda.synchronize()
    start = time.perf_counter()
    z = x @ y
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    print("CUDA tensor test: PASSED")
    print("Device:", z.device)
    print("Result shape:", tuple(z.shape))
    print(f"Matrix multiplication time: {elapsed:.4f} seconds")
    del x, y, z
    torch.cuda.empty_cache()
else:
    print("CUDA tensor test: SKIPPED")

CUDA tensor test: SKIPPED


## 8. Automatic Mixed Precision (AMP) Check

In [8]:
if torch.cuda.is_available():
    print("CUDA available:", torch.cuda.is_available())
    print("BF16 supported :", torch.cuda.is_bf16_supported())
    print("Starting recommendation: FP16 AMP unless later experiments show BF16 is preferable.")
else:
    print("AMP GPU check skipped because CUDA is unavailable.")

AMP GPU check skipped because CUDA is unavailable.


## 9. Project Folder Verification

In [9]:
required_dirs = [
    "configs", "datasets", "datasets/raw", "datasets/raw/BDD100K",
    "datasets/processed", "datasets/cyclegan", "datasets/final",
    "notebooks", "src", "models", "outputs", "logs", "results"
]
print(f"{'STATUS':<10} PATH")
print("-" * 70)
for rel in required_dirs:
    path = PROJECT_ROOT / rel
    print(f"{'OK' if path.exists() else 'MISSING':<10} {rel}")

STATUS     PATH
----------------------------------------------------------------------
OK         configs
OK         datasets
OK         datasets/raw
OK         datasets/raw/BDD100K
OK         datasets/processed
OK         datasets/cyclegan
OK         datasets/final
OK         notebooks
OK         src
OK         models
OK         outputs
OK         logs
OK         results


## 10. BDD100K Directory Check

In [10]:
bdd_root = PROJECT_ROOT / "datasets" / "raw" / "BDD100K"
image_root = bdd_root / "images" / "100k"
label_root = bdd_root / "labels"
print("BDD100K root:", bdd_root)
print("Images root :", image_root)
print("Labels root :", label_root)
for split in ["train", "val", "test"]:
    image_dir = image_root / split
    label_dir = label_root / split
    print(f"\n{split.upper()}")
    print("  Images:", image_dir.exists(), image_dir)
    print("  Labels:", label_dir.exists(), label_dir)

BDD100K root: C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K
Images root : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\images\100k
Labels root : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels

TRAIN
  Images: True C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\images\100k\train
  Labels: True C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels\train

VAL
  Images: True C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\images\100k\val
  Labels: True C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels\val

TEST
  Images: True C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\images\100k\test
  Labels: True C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\labels\test


## 11. Optional Image Count Check

This may take a little time because it scans the dataset folders.

In [11]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
def count_images(folder):
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
for split in ["train", "val", "test"]:
    print(f"{split:5}: {count_images(image_root / split):,} images")

train: 70,000 images
val  : 10,000 images
test : 20,000 images


## 12. Final Environment Status

In [12]:
project_ok = PROJECT_ROOT.exists()
core_dirs = [PROJECT_ROOT / x for x in ["datasets", "notebooks", "src", "models"]]
folders_ok = all(p.exists() for p in core_dirs)
cuda_ok = torch.cuda.is_available()
print("=" * 70)
print("ADAPTIVE ROAD VISION — ENVIRONMENT STATUS")
print("=" * 70)
print(f"Project path : {'PASS' if project_ok else 'FAIL'}")
print(f"Core folders : {'PASS' if folders_ok else 'FAIL'}")
print(f"CUDA / GPU   : {'PASS' if cuda_ok else 'FAIL'}")
if project_ok and folders_ok and cuda_ok:
    print("\nSTATUS: READY FOR GPU-BASED DEVELOPMENT")
    print("Next notebook: 02_bdd100k_dataset_verification.ipynb")
elif project_ok and folders_ok:
    print("\nSTATUS: PROJECT STRUCTURE READY; GPU/CUDA NEEDS ATTENTION")
else:
    print("\nSTATUS: CHECK PROJECT PATH/FOLDERS")

ADAPTIVE ROAD VISION — ENVIRONMENT STATUS
Project path : PASS
Core folders : PASS
CUDA / GPU   : FAIL

STATUS: PROJECT STRUCTURE READY; GPU/CUDA NEEDS ATTENTION


## Notes

- This notebook does not install or modify packages automatically.
- Later training notebooks should use CUDA, mixed precision, appropriate batch sizes, multiple data-loader workers, pinned memory where supported, and consistent experiment settings.
- Final experiment CSV/JSON result files are generated only after their relevant stages finish.